# Plastic Detector v3 — YOLO11m-seg on TrashCan + SeaClear (Colab, final run)

**Runtime:** `Runtime → Change runtime type → T4 GPU` (L4 / A100 if offered: same code, faster).

**What changes vs the current detector (YOLO11s-seg, TrashCan only, 640 px):**
- **Data:** TrashCan (same leak-free video split as the local project) **+ SeaClear** (Đuraš et al., *Scientific Data*
  11:921, 2024; 8,610 underwater ROV images from shallow Mediterranean sites, polygon masks, CC BY 4.0). SeaClear's
  categories are mapped onto the 16 TrashCan classes, so the local app keeps working unchanged.
- **Model:** YOLO11**m**-seg (2.3× the parameters) at **832 px** (small plastic items survive the downscaling).
- **Honest evaluation:** SeaClear is split **by site**: Jakljan and Slano are never seen in training and form the
  SeaClear test set (the SeaClear paper's cross-site setting). Lokrum is used for training because it holds almost all
  of SeaClear's ~980 labelled fish (clear shallow-water fish, which the current model mistakes for ROV parts). The confidence threshold is chosen on **validation** only. The current model is scored on the
  same images; the new one is kept only if it beats the current one on **validation**.

**Built to finish in one go:** a time budget (`TIME_HOURS`) so training ends inside a Colab session; checkpoints on
Drive with **automatic resume** (if Colab disconnects, just `Runtime → Run all` again); SeaClear is cached on Drive
after the first download.

**Upload to Google Drive folder `MyDrive/trashcan/` first** (from the local project):
1. `dataset.zip` — `data/raw/trashcan/dataset.zip` (~550 MB)
2. `split_manifest.csv` — `data/processed/split_manifest.csv`
3. `best_model.pt` — `models/plastic_detector/best_model.pt` (current model, for the comparison)

**Output** (`MyDrive/plastic_v3/`, download into the local project — the last cell prints exactly where):
`best_model.pt`, `operating_threshold.json`, `eval_metrics_test.json`, `test_bootstrap_confidence_intervals.json`,
`plastic_detector_v3_comparison.json`, training/PR-curve figures.

In [ ]:
!nvidia-smi
!pip install -q ultralytics
!apt-get -qq install -y p7zip-full unrar > /dev/null

In [ ]:
# ---- Config -------------------------------------------------------------
MODEL      = 'yolo11m-seg.pt'
IMGSZ      = 832
EPOCHS     = 80
TIME_HOURS = 9.0          # hard training budget (Ultralytics stops and finalises when reached; LR schedule adapts)
PATIENCE   = 15
BATCH      = 8            # T4 16 GB; use 16 on L4/A100
RUN_NAME   = 'yolo11m_seg_trashcan_seaclear_v3'
SEED       = 42
OLD_CONF   = 0.18         # current model's operating threshold (deployed)
OLD_IMGSZ  = 640
SEACLEAR_TEST_SITES = ['jakljan', 'slano']    # whole sites held out for testing (~545 images, ~190 plastic objects)
SEACLEAR_VAL_SITES  = []                      # validation / threshold choice use TrashCan val (the deployment domain)
SEACLEAR_MAX_SIDE   = 1280                    # SeaClear frames are 1920x1080; resized on conversion (labels are normalised)
N_BOOT = 1000
# -------------------------------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')
import os
from pathlib import Path
TC_DIR  = '/content/drive/MyDrive/trashcan'
OUT_DIR = Path('/content/drive/MyDrive/plastic_v3'); OUT_DIR.mkdir(parents=True, exist_ok=True)
SC_DRIVE = Path('/content/drive/MyDrive/seaclear'); SC_DRIVE.mkdir(parents=True, exist_ok=True)
for f in ('dataset.zip', 'split_manifest.csv', 'best_model.pt'):
    assert os.path.exists(f'{TC_DIR}/{f}'), f'Upload {f} to MyDrive/trashcan/ first (see the first cell)'
print('inputs OK')

## 1. TrashCan — same leak-free video split as the local project (COCO → YOLO-seg)

In [ ]:
import json, re, shutil, zipfile, random
import numpy as np, pandas as pd, yaml, cv2

CLASS_NAMES = ['rov','plant','animal_fish','animal_starfish','animal_shells','animal_crab',
               'animal_eel','animal_etc','trash_etc','trash_fabric','trash_fishing_gear',
               'trash_metal','trash_paper','trash_plastic','trash_rubber','trash_wood']
PLASTIC_ID, FISH_ID = CLASS_NAMES.index('trash_plastic'), CLASS_NAMES.index('animal_fish')

if not os.path.exists('/content/raw/dataset'):
    with zipfile.ZipFile(f'{TC_DIR}/dataset.zip') as z:
        z.extractall('/content/raw')
RAW = '/content/raw/dataset/material_version'
train_coco = json.load(open(f'{RAW}/instances_train_trashcan.json'))
val_coco = json.load(open(f'{RAW}/instances_val_trashcan.json'))
assert [c['name'] for c in train_coco['categories']] == CLASS_NAMES
tc_cat = {c['id']: i for i, c in enumerate(train_coco['categories'])}

images_by_name, anns_by_name = {}, {}
for coco, img_dir in [(train_coco, f'{RAW}/train'), (val_coco, f'{RAW}/val')]:
    id_to_name = {}
    for img in coco['images']:
        images_by_name[img['file_name']] = dict(img, path=Path(img_dir) / img['file_name'])
        id_to_name[img['id']] = img['file_name']
    for a in coco['annotations']:
        anns_by_name.setdefault(id_to_name[a['image_id']], []).append(a)

manifest = pd.read_csv(f'{TC_DIR}/split_manifest.csv')
assignment = dict(zip(manifest['file_name'], manifest['split']))
assert all(n in images_by_name for n in assignment), 'manifest images missing from dataset.zip'
vid = manifest['file_name'].str.extract(r'vid_(\d+)_')[0]
assert (manifest.assign(vid=vid).groupby('vid')['split'].nunique() == 1).all(), 'a video appears in two splits'
print(manifest['split'].value_counts().to_dict(), '- no video shared between splits')

def poly_area(p):
    xs, ys = p[0::2], p[1::2]
    return abs(sum(xs[i]*ys[(i+1)%len(xs)] - xs[(i+1)%len(xs)]*ys[i] for i in range(len(xs)))) / 2

def coco_to_yolo_lines(anns, w, h, cat_map):
    lines = []
    for a in anns:
        cls = cat_map.get(a['category_id'])
        if cls is None:
            continue
        seg = a.get('segmentation')
        parts = [p for p in (seg or []) if isinstance(p, list) and len(p) >= 6]
        if not parts:
            x, y, bw, bh = a['bbox']                      # no polygon: use the box as a 4-point polygon
            if bw <= 0 or bh <= 0:
                continue
            parts = [[x, y, x + bw, y, x + bw, y + bh, x, y + bh]]
        poly = max(parts, key=poly_area)
        norm = []
        for i in range(0, len(poly), 2):
            norm += [f"{min(max(poly[i]/w,0),1):.6f}", f"{min(max(poly[i+1]/h,0),1):.6f}"]
        lines.append(f"{cls} " + ' '.join(norm))
    return lines

DATA = Path('/content/data')
for ds_name in ('trashcan', 'seaclear'):
    for s in ('train', 'val', 'test'):
        (DATA / ds_name / s / 'images').mkdir(parents=True, exist_ok=True)
        (DATA / ds_name / s / 'labels').mkdir(parents=True, exist_ok=True)
for name, split in assignment.items():
    img = images_by_name[name]
    shutil.copyfile(img['path'], DATA / 'trashcan' / split / 'images' / name)
    (DATA / 'trashcan' / split / 'labels' / f'{Path(name).stem}.txt').write_text(
        '\n'.join(coco_to_yolo_lines(anns_by_name.get(name, []), img['width'], img['height'], tc_cat)))
print({s: len(list((DATA / 'trashcan' / s / 'images').iterdir())) for s in ('train', 'val', 'test')})

## 2. SeaClear — download (cached on Drive), map categories to TrashCan classes, split by site

In [ ]:
SC_URL = 'https://data.4tu.nl/file/4f1dff25-e157-4399-a5d4-478055461689/e1240a2e-915e-4858-93ab-c004b26b5a5f'
SC_MD5 = '1cfcf0c2fa3ef0dc219a66f063c2fe99'
rar = SC_DRIVE / 'seaclear.rar'
if not rar.exists() or rar.stat().st_size < 1.7e9:
    !wget -q --show-progress -O /content/seaclear.rar "{SC_URL}"
    shutil.copyfile('/content/seaclear.rar', rar)
md5 = !md5sum "{rar}"
assert md5[0].split()[0] == SC_MD5, f'SeaClear archive checksum mismatch: {md5}'
SC_RAW = Path('/content/seaclear_raw')
if not SC_RAW.exists():
    !7z x -y -o{SC_RAW} "{rar}" > /dev/null || unrar x -o+ "{rar}" {SC_RAW}/ > /dev/null
jsons = sorted(SC_RAW.rglob('*.json'), key=lambda p: p.stat().st_size, reverse=True)
sc_json = next(p for p in jsons if 'annotations' in json.load(open(p)))
sc = json.load(open(sc_json))
print('annotation file:', sc_json.relative_to(SC_RAW), '|', len(sc['images']), 'images |', len(sc['annotations']), 'annotations')

In [ ]:
SEACLEAR_TO_TRASHCAN = {
    # debris -> TrashCan material classes (TrashCan has no glass / ceramic / clay class: those go to trash_etc)
    'bottle_plastic': 'trash_plastic', 'bag_plastic': 'trash_plastic', 'container_plastic': 'trash_plastic',
    'cup_plastic': 'trash_plastic', 'pipe_plastic': 'trash_plastic', 'tarp_plastic': 'trash_plastic',
    'sanitaries_plastic': 'trash_plastic', 'snack_wrapper_plastic': 'trash_plastic', 'lid_plastic': 'trash_plastic',
    'rope_plastic': 'trash_plastic',
    'net_plastic': 'trash_fishing_gear',
    'can_metal': 'trash_metal', 'container_middle_size_metal': 'trash_metal', 'wreckage_metal': 'trash_metal',
    'cable_metal': 'trash_metal',
    'tire_rubber': 'trash_rubber', 'boot_rubber': 'trash_rubber',
    'rope_fiber': 'trash_fabric', 'clothing_fiber': 'trash_fabric',
    'cardboard_paper': 'trash_paper', 'snack_wrapper_paper': 'trash_paper',
    'branch_wood': 'trash_wood', 'furniture_wood': 'trash_wood',
    'bottle_glass': 'trash_etc', 'jar_glass': 'trash_etc', 'cup_ceramic': 'trash_etc', 'brick_clay': 'trash_etc',
    'tube_cement': 'trash_etc', 'unknown_instance': 'trash_etc',
    # biology
    'animal_fish': 'animal_fish', 'animal_starfish': 'animal_starfish', 'animal_shells': 'animal_shells',
    'animal_urchin': 'animal_etc', 'animal_sponge': 'animal_etc', 'animal_etc': 'animal_etc', 'plant': 'plant',
    # robot parts
    'rov_cable': 'rov', 'rov_tortuga': 'rov', 'rov_vehicle_leg': 'rov', 'rov_bluerov': 'rov',
}

sc_cats = {c['id']: c for c in sc['categories']}
unmapped = [c['name'] for c in sc['categories'] if c['name'] not in SEACLEAR_TO_TRASHCAN]
assert not unmapped, f'SeaClear categories without a mapping: {unmapped}'
sc_cat = {cid: (CLASS_NAMES.index(SEACLEAR_TO_TRASHCAN[c['name']]) if SEACLEAR_TO_TRASHCAN[c['name']] else None)
          for cid, c in sc_cats.items()}
n_ann = pd.Series([sc_cats[a['category_id']]['name'] for a in sc['annotations']]).value_counts()
print(pd.DataFrame({'annotations': n_ann, 'mapped_to': [SEACLEAR_TO_TRASHCAN[n] for n in n_ann.index]}).to_string())

In [ ]:
# Resolve image files and their site from the folder names (<site>/<camera>/<file>; file names are unique)
by_base = {}
for p in SC_RAW.rglob('*'):
    if p.suffix.lower() in ('.jpg', '.jpeg', '.png'):
        by_base.setdefault(p.name, []).append(p)
SITES = ['bistrina', 'jakljan', 'lokrum', 'marseille', 'slano']

def resolve(file_name):
    cands = by_base.get(Path(file_name.replace('\\', '/')).name, [])
    return cands[0] if len(cands) == 1 else None

sc_anns = {}
for a in sc['annotations']:
    sc_anns.setdefault(a['image_id'], []).append(a)
rows = []
for img in sc['images']:
    p = resolve(img['file_name'])
    assert p is not None, f'image not found or ambiguous: {img["file_name"]}'
    path_lower = p.as_posix().lower()
    site = next((s for s in SITES if s in path_lower), None)
    assert site, f'no site in path {p}'
    rows.append(dict(id=img['id'], path=p, site=site, w=img['width'], h=img['height']))
sc_imgs = pd.DataFrame(rows)
sc_imgs['split'] = np.where(sc_imgs.site.isin(SEACLEAR_TEST_SITES), 'test',
                   np.where(sc_imgs.site.isin(SEACLEAR_VAL_SITES), 'val', 'train'))
print(sc_imgs.groupby(['site', 'split']).size().to_string())

for r in sc_imgs.itertuples():
    im = cv2.imread(str(r.path))
    scale = min(1.0, SEACLEAR_MAX_SIDE / max(im.shape[:2]))
    if scale < 1:
        im = cv2.resize(im, (round(im.shape[1] * scale), round(im.shape[0] * scale)), interpolation=cv2.INTER_AREA)
    name = f'seaclear_{r.site}_{r.id}'
    cv2.imwrite(str(DATA / 'seaclear' / r.split / 'images' / f'{name}.jpg'), im, [cv2.IMWRITE_JPEG_QUALITY, 92])
    (DATA / 'seaclear' / r.split / 'labels' / f'{name}.txt').write_text(
        '\n'.join(coco_to_yolo_lines(sc_anns.get(r.id, []), r.w, r.h, sc_cat)))
print({s: len(list((DATA / 'seaclear' / s / 'images').iterdir())) for s in ('train', 'val', 'test')})

In [ ]:
names = {i: n for i, n in enumerate(CLASS_NAMES)}
def write_yaml(path, train, val, test=None):
    d = {'path': str(DATA), 'train': train, 'val': val, 'names': names}
    if test:
        d['test'] = test
    yaml.safe_dump(d, open(path, 'w'), sort_keys=False)
    return path
TRAIN_YAML = write_yaml(DATA / 'train.yaml', ['trashcan/train/images', 'seaclear/train/images'],
                        ['trashcan/val/images'] + (['seaclear/val/images'] if SEACLEAR_VAL_SITES else []))
TC_YAML = write_yaml(DATA / 'trashcan.yaml', 'trashcan/train/images', 'trashcan/val/images', 'trashcan/test/images')
SC_YAML = write_yaml(DATA / 'seaclear.yaml', 'seaclear/train/images', 'seaclear/test/images', 'seaclear/test/images')
print('train images:', sum(len(list((DATA / d / 'train' / 'images').iterdir())) for d in ('trashcan', 'seaclear')))

## 3. Train YOLO11m-seg (auto-resumes from Drive after a disconnect)

In [ ]:
from ultralytics import YOLO
PROJECT = str(OUT_DIR / 'runs')
run_dir = Path(PROJECT) / RUN_NAME
last, best, done = run_dir / 'weights' / 'last.pt', run_dir / 'weights' / 'best.pt', run_dir / 'DONE'

if done.exists():
    print('Training already finished earlier - skipping to evaluation.')
elif last.exists():
    print('Resuming from', last)
    YOLO(str(last)).train(resume=True)
else:
    YOLO(MODEL).train(
        data=str(TRAIN_YAML), imgsz=IMGSZ, batch=BATCH, epochs=EPOCHS, time=TIME_HOURS, patience=PATIENCE,
        device=0, seed=SEED, cos_lr=True, close_mosaic=10, workers=4, cache=False,
        project=PROJECT, name=RUN_NAME, exist_ok=True, plots=True,
    )
done.write_text('ok')
print('best weights:', best)

## 4. Evaluation — current vs new model on the same images
Plastic precision / recall / F1 by matching boxes (IoU ≥ 0.5, same class) at every confidence threshold.
The new model's threshold is chosen on **TrashCan validation**; the current model keeps its deployed 0.18.

In [ ]:
def load_gt(split_dir):
    gt = {}
    for lbl in (split_dir / 'labels').glob('*.txt'):
        img = next((split_dir / 'images').glob(lbl.stem + '.*'))
        h, w = cv2.imread(str(img)).shape[:2]
        boxes = []
        for ln in lbl.read_text().splitlines():
            v = ln.split()
            if len(v) < 7:
                continue
            xs = np.array(v[1::2], float) * w; ys = np.array(v[2::2], float) * h
            boxes.append((int(v[0]), xs.min(), ys.min(), xs.max(), ys.max()))
        gt[lbl.stem] = (str(img), boxes)
    return gt

def predict_all(weights, gt, imgsz):
    m = YOLO(str(weights)); out = {}
    for stem, (path, _) in gt.items():
        r = m.predict(path, conf=0.01, imgsz=imgsz, device=0, verbose=False)[0]
        out[stem] = [(int(c), float(s), *b) for c, s, b in zip(r.boxes.cls.tolist(), r.boxes.conf.tolist(), r.boxes.xyxy.tolist())]
    return out

def iou(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    return inter / max(1e-9, (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter)

def counts_per_image(gt, preds, cls, conf):
    # (tp, fp, fn) per image for one class at one threshold; greedy matching by confidence
    res = {}
    for stem, (_, gboxes) in gt.items():
        g = [b[1:] for b in gboxes if b[0] == cls]
        p = sorted([x for x in preds.get(stem, []) if x[0] == cls and x[1] >= conf], key=lambda x: -x[1])
        used, tp = set(), 0
        for x in p:
            best_j, best_iou = None, 0.5
            for j, gb in enumerate(g):
                if j not in used and iou(x[2:], gb) >= best_iou:
                    best_j, best_iou = j, iou(x[2:], gb)
            if best_j is not None:
                used.add(best_j); tp += 1
        res[stem] = (tp, len(p) - tp, len(g) - tp)
    return res

def prf(c):
    tp, fp, fn = (sum(v[i] for v in c.values()) for i in range(3))
    p = tp / max(1, tp + fp); r = tp / max(1, tp + fn)
    return p, r, (2 * p * r / max(1e-9, p + r)), tp, fp, fn

def sweep(gt, preds, cls, grid=np.round(np.arange(0.05, 0.81, 0.01), 2)):
    return pd.DataFrame([dict(conf=t, **dict(zip(['precision', 'recall', 'f1', 'tp', 'fp', 'fn'],
                         prf(counts_per_image(gt, preds, cls, t))))) for t in grid])

def video_of(stem):
    m = re.search(r'vid_(\d+)_', stem)
    return m.group(1) if m else stem

def bootstrap(gt, preds, cls, conf, n=N_BOOT, seed=0):
    c = counts_per_image(gt, preds, cls, conf)
    vids = {}
    for stem, v in c.items():
        vids.setdefault(video_of(stem), []).append(v)
    keys = list(vids); rng = np.random.default_rng(seed); stats = []
    for _ in range(n):
        sample = [v for k in rng.choice(keys, len(keys)) for v in vids[k]]
        stats.append(prf({i: v for i, v in enumerate(sample)})[:3])
    lo, hi = np.percentile(stats, [2.5, 97.5], axis=0)
    p, r, f = prf(c)[:3]
    fmt = lambda x, a, b: f'{x:.3f} [{a:.3f}, {b:.3f}]'
    return {'threshold': float(conf), 'precision': fmt(p, lo[0], hi[0]), 'recall': fmt(r, lo[1], hi[1]),
            'f1': fmt(f, lo[2], hi[2]), 'n_videos': len(keys)}

OLD = f'{TC_DIR}/best_model.pt'
gts = {k: load_gt(DATA / ds / s) for k, (ds, s) in
       {'tc_val': ('trashcan', 'val'), 'tc_test': ('trashcan', 'test'), 'sc_test': ('seaclear', 'test')}.items()}
preds = {(m, k): predict_all(w, gts[k], z) for m, w, z in [('old', OLD, OLD_IMGSZ), ('new', best, IMGSZ)] for k in gts}
print('predictions done')

In [ ]:
val_new = sweep(gts['tc_val'], preds[('new', 'tc_val')], PLASTIC_ID)
val_old = sweep(gts['tc_val'], preds[('old', 'tc_val')], PLASTIC_ID)
NEW_CONF = float(val_new.loc[val_new.f1.idxmax(), 'conf'])
new_val_f1 = float(val_new.f1.max())
old_val_f1 = float(val_old.loc[(val_old.conf - OLD_CONF).abs().idxmin(), 'f1'])
KEEP_NEW = new_val_f1 > old_val_f1
print(f'TrashCan VAL plastic F1: new {new_val_f1:.3f} at conf {NEW_CONF} | current {old_val_f1:.3f} at conf {OLD_CONF}')
print('DECISION (on validation):', 'use the NEW model' if KEEP_NEW else 'KEEP the current model')

import matplotlib.pyplot as plt
plt.figure(figsize=(6, 4))
for d, lab in [(val_old, 'current'), (val_new, 'new v3')]:
    plt.plot(d.conf, d.f1, label=lab)
plt.axvline(NEW_CONF, ls='--', c='gray'); plt.xlabel('confidence threshold'); plt.ylabel('plastic F1 (TrashCan val)')
plt.legend(); plt.tight_layout(); plt.savefig(OUT_DIR / 'plastic_f1_vs_conf_val.png', dpi=120)

In [ ]:
def at(model, split, cls, conf):
    p, r, f, tp, fp, fn = prf(counts_per_image(gts[split], preds[(model, split)], cls, conf))
    return {'conf': conf, 'precision': round(p, 3), 'recall': round(r, 3), 'f1': round(f, 3), 'tp': tp, 'fp': fp, 'fn': fn}

def fish_conf(model):
    s = sweep(gts['tc_val'], preds[(model, 'tc_val')], FISH_ID)
    return float(s.loc[s.f1.idxmax(), 'conf'])

ops = {'old': OLD_CONF, 'new': NEW_CONF}
table = []
for split, label in [('tc_test', 'TrashCan TEST (deep sea, unseen videos)'), ('sc_test', 'SeaClear TEST (unseen sites: ' + ', '.join(SEACLEAR_TEST_SITES) + ')')]:
    for m in ('old', 'new'):
        table.append({'split': label, 'model': 'current (YOLO11s-seg, TrashCan)' if m == 'old' else 'v3 (YOLO11m-seg, TrashCan+SeaClear)',
                      **{f'plastic_{k}': v for k, v in at(m, split, PLASTIC_ID, ops[m]).items()},
                      **{f'fish_{k}': v for k, v in at(m, split, FISH_ID, fish_conf(m)).items() if k in ('conf', 'precision', 'recall', 'f1')}})
cmp = pd.DataFrame(table)
print(cmp.to_string(index=False))
boot = {m: bootstrap(gts['tc_test'], preds[(m, 'tc_test')], PLASTIC_ID, ops[m]) for m in ('old', 'new')}
print('\nTrashCan TEST plastic, 95% CI (cluster bootstrap over videos):')
for m, b in boot.items():
    print(f'  {m}: P {b["precision"]}  R {b["recall"]}  F1 {b["f1"]}')

## 5. Standard metrics (mAP, per class) for the chosen model and save everything to Drive

In [ ]:
def ultra_eval(weights, data_yaml, imgsz, name, note):
    m = YOLO(str(weights)).val(data=str(data_yaml), split='test', imgsz=imgsz, device=0,
                               project=str(OUT_DIR / 'eval'), name=name, exist_ok=True, plots=True)
    out = {'run': name, 'weights': 'best_model.pt', 'split': 'test', 'note': note, 'leak_free': True,
           'box_precision': float(m.box.mp), 'box_recall': float(m.box.mr), 'box_map50': float(m.box.map50),
           'box_map50_95': float(m.box.map)}
    if getattr(m, 'seg', None) is not None:
        out.update({'mask_precision': float(m.seg.mp), 'mask_recall': float(m.seg.mr),
                    'mask_map50': float(m.seg.map50), 'mask_map50_95': float(m.seg.map)})
    out['per_class'] = []
    for k, c in enumerate(m.box.ap_class_index):
        p, r, ap50, ap = m.box.class_result(k)
        out['per_class'].append({'class': CLASS_NAMES[int(c)], 'precision': round(float(p), 3), 'recall': round(float(r), 3),
                                 'map50': round(float(ap50), 3), 'map50_95': round(float(ap), 3)})
        if int(c) == PLASTIC_ID:
            out.update({'plastic_precision': float(p), 'plastic_recall': float(r), 'plastic_map50': float(ap50), 'plastic_map50_95': float(ap)})
    return out

new_tc = ultra_eval(best, TC_YAML, IMGSZ, 'v3_trashcan_test', 'v3: YOLO11m-seg 832 px on TrashCan + SeaClear')
new_sc = ultra_eval(best, SC_YAML, IMGSZ, 'v3_seaclear_test', 'v3 on SeaClear held-out site')
old_tc = ultra_eval(OLD, TC_YAML, OLD_IMGSZ, 'current_trashcan_test', 'current: YOLO11s-seg 640 px on TrashCan')
old_sc = ultra_eval(OLD, SC_YAML, OLD_IMGSZ, 'current_seaclear_test', 'current on SeaClear held-out site')
for o in (old_tc, new_tc, old_sc, new_sc):
    print(f"{o['note']:55s} plastic mAP50 {o.get('plastic_map50', float('nan')):.3f} | all-class mAP50 {o['box_map50']:.3f}")

In [ ]:
winner = 'new' if KEEP_NEW else 'old'
summary = {
    'decision': 'v3 adopted' if KEEP_NEW else 'current model kept (v3 not better on validation)',
    'decided_on': 'TrashCan VAL plastic F1 (new at its val-chosen conf vs current at 0.18)',
    'val_plastic_f1': {'new': round(new_val_f1, 3), 'current': round(old_val_f1, 3)},
    'operating_conf': {'new': NEW_CONF, 'current': OLD_CONF},
    'test_table': cmp.to_dict(orient='records'),
    'trashcan_test_plastic_bootstrap': boot,
    'ultralytics_test': {'current_trashcan': old_tc, 'v3_trashcan': new_tc, 'current_seaclear': old_sc, 'v3_seaclear': new_sc},
    'data': {'trashcan_split': 'local split_manifest.csv (video-disjoint)', 'seaclear_test_sites': SEACLEAR_TEST_SITES,
             'seaclear_val_sites': SEACLEAR_VAL_SITES, 'seaclear_category_map': SEACLEAR_TO_TRASHCAN,
             'seaclear_citation': 'Duras et al., Scientific Data 11:921 (2024), doi:10.1038/s41597-024-03759-2, CC BY 4.0'},
    'model': {'arch': MODEL, 'imgsz': IMGSZ, 'time_budget_h': TIME_HOURS},
}
json.dump(summary, open(OUT_DIR / 'plastic_detector_v3_comparison.json', 'w'), indent=2, default=str)

chosen_conf = NEW_CONF if KEEP_NEW else OLD_CONF
shutil.copyfile(best if KEEP_NEW else OLD, OUT_DIR / 'best_model.pt')
json.dump({'conf': chosen_conf, 'chosen_on': 'TrashCan validation plastic F1' if KEEP_NEW else 'original deployed value',
           'model': summary['decision']}, open(OUT_DIR / 'operating_threshold.json', 'w'), indent=2)
json.dump(new_tc if KEEP_NEW else old_tc, open(OUT_DIR / 'eval_metrics_test.json', 'w'), indent=2)
json.dump({'n_videos': boot[winner]['n_videos'], 'n_boot': N_BOOT,
           'metrics': {'plastic_operating': boot[winner], 'plastic_018': boot['old']}},
          open(OUT_DIR / 'test_bootstrap_confidence_intervals.json', 'w'), indent=2)
for f in ('results.png', 'BoxPR_curve.png', 'MaskPR_curve.png', 'confusion_matrix_normalized.png'):
    if (run_dir / f).exists():
        shutil.copyfile(run_dir / f, OUT_DIR / f'v3_{f}')

print(summary['decision'])
print('\n'.join([
    '', 'Download from MyDrive/plastic_v3/ into the local project:',
    '  best_model.pt                              -> models/plastic_detector/best_model.pt   (replace; keep a copy of the old one)',
    '  operating_threshold.json                   -> models/plastic_detector/operating_threshold.json',
    '  eval_metrics_test.json                     -> outputs/metrics/eval_metrics_test.json',
    '  test_bootstrap_confidence_intervals.json   -> outputs/metrics/test_bootstrap_confidence_intervals.json',
    '  plastic_detector_v3_comparison.json        -> outputs/metrics/plastic_detector_v3_comparison.json',
    '  v3_*.png, plastic_f1_vs_conf_val.png       -> outputs/figures/',
    'Then send the printed comparison table above.']))